In [1]:
!pip install -U torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!pip install -U datasets transformers

Looking in indexes: https://download.pytorch.org/whl/cu118
ERROR: Could not find a version that satisfies the requirement torch==2.0.1 (from versions: 2.2.0+cu118, 2.2.1+cu118, 2.2.2+cu118, 2.3.0+cu118, 2.3.1+cu118, 2.4.0+cu118, 2.4.1+cu118, 2.5.0+cu118, 2.5.1+cu118, 2.6.0+cu118, 2.7.0+cu118, 2.7.1+cu118)
ERROR: No matching distribution found for torch==2.0.1


In [2]:
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

from sklearn.metrics import accuracy_score, f1_score

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
dataset = load_dataset("dair-ai/emotion")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
MODEL = "microsoft/MiniLM-L12-H384-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/MiniLM-L12-H384-uncased
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")

dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)

In [ ]:
def q_temperature_loss(logits, labels, q=1.0, temperature=1.0):

    logits = logits / temperature
    logits = torch.clamp(logits, -10, 10)

    idx = torch.arange(logits.size(0), device=logits.device)

    if abs(q - 1.0) < 1e-6:
        log_probs = torch.log_softmax(logits, dim=1)
        loss = -log_probs[idx, labels]
        return loss.mean()

    base = 1 + (1 - q) * logits
    base = torch.clamp(base, min=1e-8)

    exps = base ** (1 / (1 - q))
    probs = exps / (exps.sum(dim=1, keepdim=True) + 1e-8)

    p = probs[idx, labels]
    p = torch.clamp(p, 1e-8, 1.0)

    loss = -((p ** (1 - q) - 1) / (1 - q))

    return loss.mean()

In [8]:
class QTrainer(Trainer):

    def __init__(self, q, temperature, **kwargs):

        super().__init__(**kwargs)

        self.q = q
        self.temperature = temperature

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss = q_temperature_loss(
            logits,
            labels,
            q=self.q,
            temperature=self.temperature,
        )

        return (loss, outputs) if return_outputs else loss

In [9]:
def compute_metrics(pred):

    logits, labels = pred

    preds = np.argmax(logits, axis=1)

    return {

        "accuracy": accuracy_score(labels, preds),

        "macro_f1": f1_score(
            labels,
            preds,
            average="macro",
        ),

        "weighted_f1": f1_score(
            labels,
            preds,
            average="weighted",
        ),
    }

In [10]:
args = TrainingArguments(

    output_dir="results",

    learning_rate=2e-5,

    per_device_train_batch_size=64,

    per_device_eval_batch_size=64,

    num_train_epochs=5,

    weight_decay=0,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    greater_is_better=True,

    seed=42,

    logging_strategy="steps",
    logging_steps=250,

    report_to="none",
)

In [ ]:
results = []

qs = [0.4, 0.6, 0.8, 1.0]
temps = [0.5, 1.0, 2.0]

for q in qs:
    for T in temps:

        print("=" * 60)
        print(f"q = {q}, T = {T}")

        model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=6)

        trainer = QTrainer(
            model=model,
            args=args,
            train_dataset=dataset["train"],
            eval_dataset=dataset["validation"],
            compute_metrics=compute_metrics,
            q=q,
            temperature=T,
        )

        trainer.train()

        metrics = trainer.evaluate(dataset["test"])

        results.append({
            "q": q,
            "T": T,
            "accuracy": metrics["eval_accuracy"],
            "macro_f1": metrics["eval_macro_f1"],
            "weighted_f1": metrics["eval_weighted_f1"],
        })

q = 0.4, T = 0.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/MiniLM-L12-H384-uncased
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.741147,0.513886,0.669000,0.366061,0.581635
2,0.422364,0.273637,0.814000,0.617419,0.783317
3,0.255121,0.191857,0.874500,0.792813,0.870896
4,0.197731,0.166301,0.895000,0.846185,0.893908
5,0.174022,0.160769,0.897000,0.844212,0.895486


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
0.600235,0.600403,5,0.884000,0.743199,0.872977


In [ ]:
results_sorted = sorted(results, key=lambda x: x["macro_f1"], reverse=True)

print("\n" + "=" * 80)
print(f"{'q':<6} {'T':<6} {'Accuracy':<12} {'Macro F1':<12} {'Weighted F1':<12}")
print("-" * 80)
for r in results_sorted:
    print(f"{r['q']:<6.1f} {r['T']:<6.1f} {r['accuracy']:<12.4f} {r['macro_f1']:<12.4f} {r['weighted_f1']:<12.4f}")


q      T      Accuracy     Macro F1     Weighted F1 
--------------------------------------------------------------------------------
1.0    0.5    0.9170       0.8733       0.9179      
0.8    0.5    0.9165       0.8586       0.9147      
0.4    0.5    0.8985       0.8396       0.8974      
1.0    1.0    0.9040       0.8082       0.8995      
0.8    1.0    0.8975       0.7690       0.8884      
1.0    2.0    0.8840       0.7432       0.8730      
0.6    0.5    0.9005       0.7376       0.8864      
0.6    2.0    0.8825       0.7263       0.8677      
0.6    1.0    0.8910       0.7211       0.8756      
0.8    2.0    0.8830       0.7063       0.8663      
0.4    1.0    0.8675       0.6873       0.8509      
0.4    2.0    0.8525       0.6732       0.8363      
